In [18]:
import torch
import torch.nn as nn 
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

cuda


In [19]:
with open('wizard_of_oz.txt', 'r', encoding= 'utf-8') as f:
    text = f.read()

In [20]:
print("length of dataset in characters: ", len(text))
print(text[:200])

chars = sorted(set(text))
print(chars)
vocab_size = len(chars)
print(vocab_size)

length of dataset in characters:  232309
﻿  DOROTHY AND THE WIZARD IN OZ

  BY

  L. FRANK BAUM

  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ, OZMA OF OZ, ETC.

  ILLUSTRATED BY JOHN R. NEILL

  BOOKS OF WONDER WILLIAM MORROW & CO., INC. NEW
['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']
81


In [21]:
str_to_int = {char : idx for idx, char in enumerate(chars)} # creatd this dict to map chars to ints
int_to_str = {idx : char for idx, char in enumerate(chars)} # created this dict to map ints to chars

encode = lambda string: [str_to_int[c] for c in string] # its iterating char from the string and converting the char to an int based on the str to int mapping
decode = lambda lst: ''.join([int_to_str[i] for i in lst]) # its taking list of ints and returning strings

print(f"str to int: {str_to_int}")
print(f"int to str: {int_to_str}")
print("")
print(encode("Hello"))
print(decode(encode("Hello")))


str to int: {'\n': 0, ' ': 1, '!': 2, '"': 3, '&': 4, "'": 5, '(': 6, ')': 7, '*': 8, ',': 9, '-': 10, '.': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, ':': 22, ';': 23, '?': 24, 'A': 25, 'B': 26, 'C': 27, 'D': 28, 'E': 29, 'F': 30, 'G': 31, 'H': 32, 'I': 33, 'J': 34, 'K': 35, 'L': 36, 'M': 37, 'N': 38, 'O': 39, 'P': 40, 'Q': 41, 'R': 42, 'S': 43, 'T': 44, 'U': 45, 'V': 46, 'W': 47, 'X': 48, 'Y': 49, 'Z': 50, '[': 51, ']': 52, '_': 53, 'a': 54, 'b': 55, 'c': 56, 'd': 57, 'e': 58, 'f': 59, 'g': 60, 'h': 61, 'i': 62, 'j': 63, 'k': 64, 'l': 65, 'm': 66, 'n': 67, 'o': 68, 'p': 69, 'q': 70, 'r': 71, 's': 72, 't': 73, 'u': 74, 'v': 75, 'w': 76, 'x': 77, 'y': 78, 'z': 79, '\ufeff': 80}
int to str: {0: '\n', 1: ' ', 2: '!', 3: '"', 4: '&', 5: "'", 6: '(', 7: ')', 8: '*', 9: ',', 10: '-', 11: '.', 12: '0', 13: '1', 14: '2', 15: '3', 16: '4', 17: '5', 18: '6', 19: '7', 20: '8', 21: '9', 22: ':', 23: ';', 24: '?', 25: 'A', 26: 'B', 27: 'C', 28: 'D

In [22]:
data = torch.tensor(encode(text), dtype= torch.long) #tokenized or encoded the entire .txt file and then fed to pytorch to return the entire dataset as tensor
print(data[:100])

n = int(0.8 * len(data))
train_data = data[:n]
validation_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else validation_data
    idx = torch.randint(len(data) - block_size, (batch_size,))

    print(idx)

    x = torch.stack([data[i: i + block_size] for i in idx]).to(device)
    y = torch.stack([data[i+1: i + block_size + 1] for i in idx]).to(device)

    return x, y

x, y = get_batch('train')
print('inputs: ' )
print(x)
print('targets: ')
print(y)


tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])
tensor([173435, 185527,  98044,  68767])
inputs: 
tensor([[71, 23,  1, 72, 68,  1, 73, 61],
        [71, 75, 58, 57,  1, 57, 62, 71],
        [55, 58, 57,  1, 73, 61, 58,  1],
        [74, 72, 61,  1, 78, 68, 74,  1]], device='cuda:0')
targets: 
tensor([[23,  1, 72, 68,  1, 73, 61, 58],
        [75, 58, 57,  1, 57, 62, 71, 58],
        [58, 57,  1, 73, 61, 58,  1, 72],
        [72, 61,  1, 78, 68, 74,  1, 54]], device='cuda:0')


In [23]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [24]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


XE0M-YH9pOi_9mse?!8ZQ[-(Mi"jDaezvk2_'5E&E?7-y:z HT'uIV-Mdd&
b4ZP]YMz[v 0PZd&e1&"?,znzNqX-30t[E6)G6)KaO(9Y"
.c[Ia'Cn1W8cc)"?6Nk'Qt'qYB2H ckeVLSP]"?k*MD(a7Ypk!c],lsj&2_﻿sv7I44!FqIf"i
mP7FaOtaZ93(ep"R﻿.Jimqnt*rH9.XBJY(eYwku6TK44FaZ;54-JI&[1q8*V(]v 6-r*ePRCn]EBMlk.f"z0;B-6J]RkQXMbMH yrNY]1o,gwePE'.]:S5;E8.,W8z;zge79,C4,]M6T?fSoHfwv5E.lc)ap)jj4r?7
OiT4kZJz94SWvijRebNTyyg3H(7H,gk.J:NFK3gJu;z0?"pP]8PqzOKo4r1OiyryOPPPWYx4!K[1OyeNRQbtLwJuEwM6)j1nJLuf_*K6)BOOolsp(
hq6Tt_VN3"k8;v﻿oW8ePWAfg,x&
TmlksED(]Tvlg


In [25]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

tensor([ 62872, 107619,  16717,   7088])
tensor([ 26511, 181625,  16979, 183767])
tensor([124564, 152038,   6352,  88525])
tensor([108112, 152970,  59319,  25377])
tensor([ 62101, 105729, 156509, 148309])
tensor([ 45599,  96365,  50363, 172036])
tensor([18870, 17873,   790, 40961])
tensor([   826, 105727,  66114,  16838])
tensor([164602,  48707,  28144, 114690])
tensor([139935,  53592,   3516,  24131])
tensor([140128, 170100, 129740,    334])
tensor([179694,  24022,  62074, 110394])
tensor([  116, 18599, 30769, 78363])
tensor([ 88615, 180801, 151314, 175627])
tensor([ 31640, 135348,    489,   7874])
tensor([ 57567,  84052, 109116, 164170])
tensor([17722, 54520, 21774, 69726])
tensor([ 55779,  82711,  24890, 172449])
tensor([ 44066,  79178, 117640,   2373])
tensor([ 31886, 102112,  49089, 175973])
tensor([160578, 149353,  60609, 143062])
tensor([ 62865, 142883, 119932, 117893])
tensor([ 62028, 127023,  74667, 122182])
tensor([118653, 133878,  72214,  43387])
tensor([ 63075, 138273, 1850